In [ ]:
import torch
import torchvision.transforms as transforms
import onnx

In [ ]:
!pip install onnx onnxscript

In [ ]:
from google.colab import drive
model_path = '/content/drive/MyDrive/Version 1 DATASET/Models/v12/v1DataEfficentNetB5Model_v12.pt'

In [ ]:
from torchvision.models import efficientnet_b5, EfficientNet_B5_Weights
model = efficientnet_b5(weights=None)
model.classifier[1] = torch.nn.Linear(2048, 2)

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load(model_path, map_location=device))

model.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 48, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(48, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
            (1): BatchNorm2d(48, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormAct

In [ ]:
image_size = (456, 456)
transform = transforms.Compose([transforms.Resize(image_size),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                                     std=[0.229, 0.224, 0.225])])

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

In [ ]:
test_input = torch.randn(1, 3, 456, 456)

torch.onnx.export(
    model,
    test_input,
    "v1DataEfficentNetB5Model_v12.onnx",
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input' : {0 : 'batch_size'}, 'output' : {0 : 'batch_size'}}
)

/tmp/ipython-input-455670576.py:3: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0209 01:29:48.115000 1933 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0209 01:29:48.618000 1933 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, alig

[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...


[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:65: adapt: Asserti

Applied 232 of general pattern rewrite rules.


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.9.0+cu126',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input"<FLOAT,[s77,3,456,456]>
            ),
            outputs=(
                %"output"<FLOAT,[1,2]>
            ),
            initializers=(
                %"features.1.0.block.0.0.weight"<FLOAT,[48,1,3,3]>{Tensor(...)},
                %"features.1.0.block.1.fc1.weight"<FLOAT,[12,48,1,1]>{TorchTensor(...)},
                %"features.1.0.block.1.fc1.bias"<FLOAT,[12]>{TorchTensor(...)},
                %"features.1.0.block.1.fc2.weight"<FLOAT,[48,12,1,1]>{TorchTensor(...)},
                %"features.1.0.block.1.fc2.bias"<FLOAT,[48]>{TorchTensor(...)},
                %"features.1.1.block.0.0.weight"<FLOAT,[24,1,3,3]>{Tensor(...)},
                %"featur

In [ ]:
model_check = onnx.load("v1DataEfficentNetB5Model_v12.onnx")
onnx.checker.check_model(model_check)
print("Model is valid and weights are linked!")

Model is valid and weights are linked!
